https://docs.coingecko.com/reference/coins-markets


In [1]:
import requests
import json
import pandas as pd
from datetime import datetime
import os
from dotenv import load_dotenv
import traceback

import logging
log_file = '/content/drive/MyDrive/Estudos/Python/Dados/Bitcoin/logs/pipeline.log'

logging.basicConfig(
    filename=log_file,
    filemode='a',
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO
)

pd.options.display.float_format = '{:.2f}'.format
pd.set_option('display.max_columns', None)

In [2]:
# Extract

def extract_data(url, params, headers):
  # Recebe os dados da api
  response = requests.get(url, params=params, headers=headers)

  if response.status_code == 200:
    # Transforma a resposta da api em json
    data = response.json()

  else:
    raise Exception(f'Erro ao acessar API: {response.status_code}')

  # Transforma o json em um df(dataframe)
  df = pd.DataFrame(data)

  return df

In [3]:
# Transform

def transform_data(df):
  # Cria a nova coluna "Data da coleta" com a data e hora em que os dados foram coletados
  df['data_da_coleta'] = datetime.now().replace(second=0, microsecond=0)

  # Mudando o tipo da coluna
  df['market_cap'] = df['market_cap'].astype(float)

  # Filtrando para pegar só as colunas importantes e dpois tirando o que é nulo
  df_final = df[['id', 'name', 'current_price', 'market_cap', 'price_change_percentage_24h', 'data_da_coleta']]
  df_final = df_final.dropna()

  # Renomeando as colunas
  df_final = df_final.rename(columns={
      'name': 'nome',
      'current_price': 'preco_atual',
      'market_cap': 'capitalizacao_mercado',
      'price_change_percentage_24h': 'variacao_preco_24h'
  })

  return df_final

In [4]:
# Load

def load_data(df, caminho_csv):
  # Verifica se existe o arquivo do argumento
  if os.path.exists(caminho_csv):
    df_existente = pd.read_csv(caminho_csv)

    # Concatena e remove o que é duplicata
    df = pd.concat([df_existente, df])
    df = df.drop_duplicates(subset=['id', 'data_da_coleta'], keep= 'last')

  # Salva o arquivo
  df.to_csv(caminho_csv, index=False)

In [5]:
# Main

load_dotenv()
api_key = os.getenv('API_KEY')

url = 'https://api.coingecko.com/api/v3/coins/markets'
params = {
    'vs_currency': 'usd',
    'order': 'market_cap_desc',
    'per_page': 20,
    'page': 1
}
headers = {'x_cg_demo_api_key': api_key}

caminho_csv_bruto = '/content/drive/MyDrive/Estudos/Python/Dados/Bitcoin/data/raw/bitcoin_bruto.csv'
caminho_csv_tratado = '/content/drive/MyDrive/Estudos/Python/Dados/Bitcoin/data/processed/bitcoin_tratado.csv'


def main():
  try:
    logging.info('===== INICIANDO PIPELINE =====')
    ## ETAPA 1 - Extract
    logging.info('Iniciando a extração dos dados')
    df_bruto = extract_data(url, params, headers)

    if df_bruto.empty:
      logging.warning('DataFrame está vazio')
      return

    logging.info('Extração finalizada')

    ## ETAPA 2 - Transform
    logging.info('Iniciando a transformação dos dados')
    df_tratado = transform_data(df_bruto)
    logging.info('Transformação finalizada')

    ## ETAPA 3 - Load
    logging.info('Iniciando o carregamento dos dados')
    load_data(df_bruto, caminho_csv_bruto)
    load_data(df_tratado, caminho_csv_tratado)
    logging.info('Carregamento finalizado')

    logging.info('===== PIPELINE FINALIZADO =====')
  except Exception as e:
    logging.error(f'Erro: {e}')
    logging.error(traceback.format_exc())

main()